# VietMed-NER — PhoBERT vs ViHealthBERT (Softmax, full fine-tune) + Gradio

Reuse the PhoBERT Softmax checkpoint, full fine-tune ViHealthBERT with the **same config**,
compare entity-level F1 + error analysis, then a Gradio demo with an encoder picker.
Spec: `docs/superpowers/specs/2026-06-30-phobert-vs-vihealthbert-softmax-gradio-design.md`

In [ ]:
!pip -q install "transformers==4.44.2" "datasets==2.21.0" seqeval pytorch-crf py_vncorenlp pandas gradio
# §0 Setup
import os, random, numpy as np, torch
def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(42)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR   = '/content/drive/MyDrive/vietmed_ner'
CKPT_DIR    = f'{DRIVE_DIR}/checkpoints'
RESULTS_DIR = f'{DRIVE_DIR}/results'
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
# §1 Config — identical training config for both encoders (only the encoder differs)
CFG_PHO = dict(encoder="vinai/phobert-base",                head="softmax",
               segment=True, max_len=256, batch_size=16,
               ckpt=f'{CKPT_DIR}/01_phobert_softmax.pt.best')      # REUSE (no retrain)
CFG_VIH = dict(encoder="demdecuong/vihealthbert-base-word", head="softmax",
               segment=True, max_len=256, batch_size=16,
               ckpt=f'{CKPT_DIR}/02_vihealthbert_softmax.pt')      # TRAIN + save here
LR, EPOCHS, PATIENCE = 2e-5, 30, 3

In [ ]:
# §2 Data
from datasets import load_dataset
raw = load_dataset("leduckhai/VietMed-NER")
for c in ("audio", "duration"):
    if c in raw["train"].column_names:
        raw = raw.remove_columns(c)
TAG_COL  = "labels" if "labels" in raw["train"].column_names else "tags"
WORD_COL = "words"
def get_words_labels(split):
    ds = raw[split]; return list(ds[WORD_COL]), list(ds[TAG_COL])
_all_tags = set(t for split in raw for row in raw[split][TAG_COL] for t in row)
LABEL_LIST = ["O"] + sorted(t for t in _all_tags if t != "O")
label2id = {t:i for i,t in enumerate(LABEL_LIST)}
id2label = {i:t for t,i in label2id.items()}
print(len(LABEL_LIST), "labels; splits:", {k: len(raw[k]) for k in raw})

In [ ]:
# §3 Models
pass